# Load both datasets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import files
uploaded = files.upload()

Saving E0.xlsx to E0.xlsx


In [3]:
epl = pd.read_excel("E0.xlsx")


In [4]:
from google.colab import files
uploaded = files.upload()

Saving D1.xlsx to D1.xlsx


In [5]:
bundesliga = pd.read_excel("D1.xlsx")

# Add league labels

In [6]:
epl["League"] = "Premier League"
bundesliga["League"] = "Bundesliga"

# Combine datasets

In [7]:
combined = pd.concat(
    [epl, bundesliga],
    ignore_index=True
)

In [8]:
combined.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA,League
0,E0,2025-08-15,20:00:00,Liverpool,Bournemouth,4,2,H,1,0,...,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86,Premier League
1,E0,2025-08-16,12:30:00,Aston Villa,Newcastle,0,0,D,0,0,...,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86,Premier League
2,E0,2025-08-16,15:00:00,Brighton,Fulham,1,1,D,0,0,...,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08,Premier League
3,E0,2025-08-16,15:00:00,Sunderland,West Ham,3,0,H,0,0,...,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97,Premier League
4,E0,2025-08-16,15:00:00,Tottenham,Burnley,3,0,H,1,0,...,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92,Premier League


# Convert odds into probabilities

In [9]:
combined["home_prob"] = 1 / combined["B365H"]
combined["draw_prob"] = 1 / combined["B365D"]
combined["away_prob"] = 1 / combined["B365A"]

# Normalize probabilities

In [10]:
total = (
    combined["home_prob"] +
    combined["draw_prob"] +
    combined["away_prob"]
)

combined["home_prob"] /= total
combined["draw_prob"] /= total
combined["away_prob"] /= total

# Encode actual match outcomes

In [11]:
combined["actual_home"] = (
    combined["FTR"] == "H"
).astype(int)

combined["actual_draw"] = (
    combined["FTR"] == "D"
).astype(int)

combined["actual_away"] = (
    combined["FTR"] == "A"
).astype(int)

# Do bookmakers systematically overvalue home advantage in football?

# Home Win Rate
Business Question:

How often do home teams actually win?

In [12]:
combined["home_win"] = (
    combined["FTR"] == "H"
).astype(int)

home_win_rate = combined.groupby("League")[
    "home_win"
].mean()

home_win_rate

,home_win
League,
Bundesliga,0.430556
Premier League,0.426934


# Home Goals vs Away Goals
Question:

Do home teams score significantly more?

In [13]:
goal_analysis = combined.groupby("League").agg(
    avg_home_goals=("FTHG", "mean"),
    avg_away_goals=("FTAG", "mean")
)

goal_analysis

,avg_home_goals,avg_away_goals
League,,
Bundesliga,1.756944,1.458333
Premier League,1.524355,1.237822


# Bookmaker Home Probability
Question:

How strongly do bookmakers favor home teams?

In [14]:
home_prob_analysis = combined.groupby("League").agg(
    avg_home_prob=("home_prob", "mean")
)

home_prob_analysis

,avg_home_prob
League,
Bundesliga,0.444904
Premier League,0.436370


# Home Market Calibration
Question:

When bookmakers heavily favor home teams… are they correct?

# Home Overperformance
Question:

Which teams outperform expectations at home?

In [15]:
combined["home_overperf"] = (
    combined["actual_home"] -
    combined["home_prob"]
)

home_team_advantage = combined.groupby(
    "HomeTeam"
)["home_overperf"].mean()

home_team_advantage.sort_values(
    ascending=False
)

,home_overperf
HomeTeam,
Fulham,0.199133
Sunderland,0.161760
Aston Villa,0.148787
Man United,0.146656
Stuttgart,0.145599
Dortmund,0.136939
RB Leipzig,0.130585
Arsenal,0.101558
Hoffenheim,0.082127
